# TD — Construction de la collection `entreprise`

On dispose de l'export KBO Open Data (8 fichiers CSV) monté sur le volume Docker,
et d'une base MongoDB vide. L'objectif de ce TD est de construire, fonction par
fonction, le pipeline qui charge ces CSV dans MongoDB puis les joint pour produire
une collection `entreprise` : un document par entreprise, avec ses établissements
et ses branches imbriqués.

**Environnement** : ce notebook est prévu pour tourner à côté du `docker-compose.yml`
fourni (service `mongo` exposé sur `localhost:27017`, utilisable directement avec
MongoDB Compass). Décompressez l'archive KBO Open Data dans `./data/kbo` (à côté du
`docker-compose.yml`) avant de lancer les cellules ci-dessous.


In [4]:
%pip install pymongo

     |████████████████████████████████| 763 kB 9.4 MB/s eta 0:00:01
  Using cached dnspython-2.7.0-py3-none-any.whl (313 kB)
You should consider upgrading via the '/Users/chancybayedi-mayombo/Downloads/projet final partie I /.venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [10]:
import os
import csv
from pathlib import Path
from pprint import pprint
from __future__ import annotations
import pymongo

# Ces valeurs peuvent être surchargées par des variables d'environnement,
# ce qui permet d'utiliser ce notebook aussi bien en local (Compass + Jupyter
# sur l'hôte) que depuis le service "jupyter" du docker-compose fourni.
DATA_DIR = Path(os.environ.get("DATA_DIR", "../data/kbo"))
MONGO_URI = os.environ.get("MONGO_URI", "mongodb://localhost:27017")
DB_NAME = os.environ.get("MONGO_DB", "kbo")

client = pymongo.MongoClient(MONGO_URI)
db = client[DB_NAME]


## 1) Chargement d'un CSV dans une collection

Les fichiers de l'export sont volumineux.

Écrivez une fonction qui charge un fichier CSV donné dans une
collection MongoDB donnée par batch.

Appliquez cette fonction aux 8 fichiers de l'export pour peupler 8 collections,
une par fichier.


In [11]:
def load_csv_to_collection(csv_path: Path, collection, batch_size: int = 5000, id_field: str | None = None):
    """Charge un fichier CSV dans une collection MongoDB, par lots de `batch_size` lignes.

    Si `id_field` est fourni, la valeur de ce champ est utilisée comme `_id` du
    document (utile pour l'entité "entreprise", dont on veut que le `_id` du
    document final soit le numéro d'entreprise). Sinon, MongoDB génère un
    ObjectId automatiquement.
    """
    collection.delete_many({})  # on repart d'une collection vide, le script est ré-exécutable

    batch = []
    with open(csv_path, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            doc = dict(row)
            if id_field:
                doc["_id"] = doc[id_field]
            batch.append(doc)
            if len(batch) >= batch_size:
                collection.insert_many(batch)
                batch = []
    if batch:
        collection.insert_many(batch)


# Les 8 fichiers de l'export, et le champ à utiliser comme _id le cas échéant.
# Seul enterprise.csv a besoin d'un _id explicite : c'est le seul niveau dont le
# _id du document final doit être un identifiant métier stable (EnterpriseNumber).
FILES = [
    ("enterprise.csv",    "enterprise",    "EnterpriseNumber"),
    ("establishment.csv", "establishment", None),
    ("denomination.csv",  "denomination",  None),
    ("address.csv",       "address",       None),
    ("contact.csv",       "contact",       None),
    ("activity.csv",      "activity",      None),
    ("branch.csv",         "branch",       None),
    ("code.csv",           "code",         None),
]

for filename, collection_name, id_field in FILES:
    load_csv_to_collection(DATA_DIR / filename, db[collection_name], id_field=id_field)
    print(f"{collection_name:15s} <- {filename:22s} ({db[collection_name].count_documents({}):,} documents)")


enterprise      <- enterprise.csv         (1,955,776 documents)
establishment   <- establishment.csv      (1,691,048 documents)
denomination    <- denomination.csv       (3,354,096 documents)
address         <- address.csv            (2,886,648 documents)
contact         <- contact.csv            (708,661 documents)
activity        <- activity.csv           (34,373,618 documents)
branch          <- branch.csv             (7,319 documents)
code            <- code.csv               (21,468 documents)


## 2) Le schéma de la jointure

Indexez dans chaque collection concernée le ou les champs qui serviront de clé de jointure.


In [12]:
# Clé primaire de l'entreprise elle-même (utilisée aussi comme _id, cf. question 1)
db.enterprise.create_index("EnterpriseNumber", unique=True)

# Clés étrangères vers l'entreprise
db.establishment.create_index("EnterpriseNumber")
db.branch.create_index("EnterpriseNumber")

# Clé primaire des établissements et des succursales, utilisée comme clé de
# jointure du côté "détails" (denomination/address/contact/activity)
db.establishment.create_index("EstablishmentNumber")
db.branch.create_index("Id")

# Clé de jointure commune aux 4 collections de détails, quel que soit le
# niveau d'entité (entreprise, établissement, branche)
db.denomination.create_index("EntityNumber")
db.address.create_index("EntityNumber")
db.contact.create_index("EntityNumber")
db.activity.create_index("EntityNumber")


'EntityNumber_1'

## 3) Rejoindre les détails d'une entité

Trois niveaux d'entités — entreprise, établissement, branche — ont chacun
besoin des mêmes quatre informations complémentaires : leurs dénominations,
leurs adresses, leurs contacts et leurs activités. Ces quatre informations
vivent chacune dans leur propre collection, et s'y rattachent toujours de la
même manière, quel que soit le type d'entité concerné.

Écrivez une fonction réutilisable qui, étant donné le nom du champ à utiliser
comme clé de jointure du côté de l'entité courante, construit les étapes
d'agrégation nécessaires pour rattacher ces quatre informations. Cette fonction
sera appelée trois fois dans la suite du TD, une fois par niveau d'entité, avec
à chaque fois un nom de champ différent.


In [13]:
def _detail_lookups(primary_key: str) -> list[dict]:
    """Étapes d'agrégation qui rattachent denominations/addresses/contacts/activities
    à l'entité courante, en joignant `primary_key` (côté entité) sur `EntityNumber`
    (côté collection de détail).

    `primary_key` est le nom du champ de l'entité courante qui identifie cette
    entité dans les collections de détail : "EnterpriseNumber" pour une
    entreprise, "EstablishmentNumber" pour un établissement, "Id" pour une
    branche.
    """
    detail_collections = [
        ("denomination", "denominations"),
        ("address",      "addresses"),
        ("contact",       "contacts"),
        ("activity",      "activities"),
    ]
    return [
        {
            "$lookup": {
                "from": collection_name,
                "localField": primary_key,
                "foreignField": "EntityNumber",
                "as": alias,
            }
        }
        for collection_name, alias in detail_collections
    ]


## 4) Rejoindre les succursales

Une succursale représente la présence en Belgique d'une entreprise étrangère.
Chaque succursale est rattachée à une entreprise, et a, comme vu à la question
3, ses propres dénominations et adresses (mais jamais de contacts ni
d'activités).

Écrivez une fonction qui construit l'étape d'agrégation permettant de rattacher,
pour chaque entreprise, la liste de ses succursales complètes, chaque
succursale devant elle-même déjà porter ses propres dénominations et adresses,
obtenues via la fonction de la question 3. Le lien entre une entreprise et ses
succursales ne se fait pas sur le même champ que celui utilisé à l'intérieur
d'une succursale pour aller chercher ses propres dénominations et adresses :
il faudra donc corréler explicitement les deux niveaux.


In [14]:
def _branches_lookup() -> dict:
    """Étape d'agrégation qui rattache à chaque entreprise la liste de ses
    succursales, chaque succursale portant déjà ses propres dénominations et
    adresses (obtenues via `_detail_lookups`).

    - Le lien entreprise -> succursale se fait sur EnterpriseNumber (côté
      entreprise) == EnterpriseNumber (côté branch) : d'où le `let`/`$expr`.
    - Le lien succursale -> ses propres détails se fait sur Id (côté branch)
      == EntityNumber (côté denomination/address/contact/activity) : d'où
      l'appel à `_detail_lookups("Id")`.
    Une succursale n'a jamais de contacts ni d'activités dans les données :
    on réutilise quand même `_detail_lookups` telle quelle (question 3), ces
    deux tableaux ressortiront simplement vides.
    """
    return {
        "$lookup": {
            "from": "branch",
            "let": {"enterpriseNumber": "$EnterpriseNumber"},
            "pipeline": [
                {"$match": {"$expr": {"$eq": ["$EnterpriseNumber", "$$enterpriseNumber"]}}},
                *_detail_lookups("Id"),
            ],
            "as": "branches",
        }
    }


## 5) Rejoindre les établissements

Même exercice que la question 4, mais pour les établissements, les unités
opérationnelles d'une entreprise belge.

Comme pour les succursales, chaque établissement doit déjà porter ses propres
dénominations, adresses, contacts et activités (obtenus via la fonction de la
question 3) avant d'être rattaché à son entreprise.


In [15]:
def _establishments_lookup() -> dict:
    """Étape d'agrégation qui rattache à chaque entreprise la liste de ses
    établissements, chaque établissement portant déjà ses propres
    dénominations, adresses, contacts et activités.

    - Le lien entreprise -> établissement se fait sur EnterpriseNumber (côté
      entreprise) == EnterpriseNumber (côté establishment).
    - Le lien établissement -> ses propres détails se fait sur
      EstablishmentNumber (côté establishment) == EntityNumber (côté détails).
    """
    return {
        "$lookup": {
            "from": "establishment",
            "let": {"enterpriseNumber": "$EnterpriseNumber"},
            "pipeline": [
                {"$match": {"$expr": {"$eq": ["$EnterpriseNumber", "$$enterpriseNumber"]}}},
                *_detail_lookups("EstablishmentNumber"),
            ],
            "as": "establishments",
        }
    }


## 6) Assembler et exécuter le pipeline complet

Combinez les fonctions précédentes en un seul pipeline d'agrégation, lancé sur
la collection contenant les entreprises : les quatre informations
complémentaires de l'entreprise elle-même, puis ses établissements, puis ses
succursales.

Le résultat de ce pipeline doit s'écrire dans une nouvelle collection.

Exécutez le pipeline, et affichez le document
complet d'une entreprise qui possède au moins un établissement, et d'une
entreprise qui possède au moins une succursale.


In [16]:
pipeline = [
    *_detail_lookups("EnterpriseNumber"),   # dénominations/adresses/contacts/activités de l'entreprise elle-même
    _establishments_lookup(),               # ses établissements, complets
    _branches_lookup(),                     # ses succursales, complètes
    {"$out": "entreprise"},                 # écrit le résultat dans la collection "entreprise"
]

db.enterprise.aggregate(pipeline)

print(f"entreprise: {db.entreprise.count_documents({}):,} documents")


entreprise: 1,955,776 documents


In [17]:
entreprise_avec_etablissement = db.entreprise.find_one({"establishments.0": {"$exists": True}})
pprint(entreprise_avec_etablissement)


{'EnterpriseNumber': '0200.065.765',
 'JuridicalForm': '416',
 'JuridicalFormCAC': '',
 'JuridicalSituation': '000',
 'StartDate': '09-08-1960',
 'Status': 'AC',
 'TypeOfEnterprise': '2',
 '_id': '0200.065.765',
 'activities': [{'ActivityGroup': '006',
                 'Classification': 'MAIN',
                 'EntityNumber': '0200.065.765',
                 'NaceCode': '84130',
                 'NaceVersion': '2025',
                 '_id': ObjectId('6a6794bccf46868bd6acefa5')},
                {'ActivityGroup': '001',
                 'Classification': 'MAIN',
                 'EntityNumber': '0200.065.765',
                 'NaceCode': '68121',
                 'NaceVersion': '2025',
                 '_id': ObjectId('6a6794bccf46868bd6acefa6')},
                {'ActivityGroup': '001',
                 'Classification': 'SECO',
                 'EntityNumber': '0200.065.765',
                 'NaceCode': '84130',
                 'NaceVersion': '2025',
                 '_id': Objec

In [18]:
entreprise_avec_succursale = db.entreprise.find_one({"branches.0": {"$exists": True}})
pprint(entreprise_avec_succursale)


{'EnterpriseNumber': '0257.883.408',
 'JuridicalForm': '030',
 'JuridicalFormCAC': '',
 'JuridicalSituation': '000',
 'StartDate': '01-09-1995',
 'Status': 'AC',
 'TypeOfEnterprise': '2',
 '_id': '0257.883.408',
 'activities': [],
 'addresses': [{'Box': '',
                'CountryFR': 'Turquie',
                'CountryNL': 'Turkije',
                'DateStrikingOff': '',
                'EntityNumber': '0257.883.408',
                'ExtraAddressInfo': '',
                'HouseNumber': '0',
                'MunicipalityFR': 'yenibosna - Istamboul',
                'MunicipalityNL': 'yenibosna - Istamboul',
                'StreetFR': 'itkib bis ticaret komplexi b/blok coban cesme '
                            'mekvil sanayi/caddesi',
                'StreetNL': 'itkib bis ticaret komplexi b/blok coban cesme '
                            'mekvil sanayi/caddesi',
                'TypeOfAddress': 'REGO',
                'Zipcode': '34196',
                '_id': ObjectId('6a679495cf4